# MP35: geographic splitting before tiling
A scene is a source image; a region groups related scenes. Split regions before extracting patches, so nearby samples do not leak between training and evaluation. We audit the already frozen assignments rather than selecting a new favorable test set.

For region sets T,V,E require T∩V=T∩E=V∩E=∅. For boxes A,B define g=max(Bwest−Aeast, Awest−Beast, Bsouth−Anorth, Asouth−Bnorth). Accept only g>b, where b=0.02 degrees. This is an axis-envelope rule, not Euclidean/geodesic distance. Longitude degrees shrink physically toward the poles.

Worked example: A=[0,0,1,1], B=[2,0,3,1], g=1 degree. It passes b=.02 but fails b=1 because equality is rejected.

In [ ]:
from mp35_tools import audit_geography
scenes=[dict(id='a',region_id='a',split='train',wgs84_bounds=[0,0,1,1]),dict(id='b',region_id='b',split='test',wgs84_bounds=[2,0,3,1])]
audit_geography(scenes)

Exercise: change the second region to a. Why must the audit fail even if its box is distant? Solution: region identity is an independent leakage constraint; geometry alone is insufficient.

In [ ]:
try:
    audit_geography([scenes[0],dict(scenes[1],region_id='a')])
except ValueError as error:
    print('Expected rejection:',error)
else:
    raise AssertionError('Leakage was accepted')

In [ ]:
import json
from pathlib import Path
ROOT=Path.cwd().parents[1]
report=json.loads((ROOT/'reports/mp35-geographic-splits.json').read_text())
print(report['audit']['counts'])
print(len(report['audit']['cross_split_pairs']), 'cross-split pairs')
print(report['scope'])

In [ ]:
from IPython.display import SVG,display
display(SVG(filename=str(ROOT/'reports/mp35-splits/map.svg')))

Interpretation: four train regions, four validation regions and one preserved test region pass 24 cross-split checks. No reassignment or test inference occurred. Sixteen patches from one scene remain one geographic replicate, not sixteen independent regions. Overlap rejection addresses leakage but does not establish sensor independence or broad generalization.

Exercise: if two patches from each of two scenes are split randomly, could both sets contain each scene? Yes: place the first patch in train and second in test. Scene-first assignment prevents this. Next question: how does performance transfer across sensors under a separately declared protocol? MP36 addresses that requirement.

Provenance: pinned OpenSR NAIP manifest and checksums in the report; NAIP is aerial, not satellite imagery. Dataset-card MIT declaration and USDA public-domain source attribution retained. These metadata exports reuse existing split declarations; they are not a newly selected benchmark.